In [2]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

In [3]:
# ---------------------------------------
# paths
# ---------------------------------------
# Elevation data from Vinther et al. (2009)
vinther = xr.open_dataset("../../FesmData/Vinther2009_elevations/vinther2009.nc")

# Ensemble elevations 
yelmo_elev = xr.open_dataset("../output/ensemble_elevations.nc")
yelev = yelmo_elev.sel(time=slice(-11700, -40)) # Years covered by Vinther reconstruction
vint = vinther.sel(time=yelev.time) # Remove years bellow yelmo resolution
sim0 = xr.open_dataset("/home/luciagu/projects/3_yelmo_deglaciation/ensemble_reduced/0/yelmo2D_reduced.nc")

# Ensemble GIA 
ygia=xr.open_dataset("../output/ensemble_gia.nc")

# Ensemble RSL 
yrsl = xr.open_dataset("../output/ensemble_rsl.nc")

# Present-day data
bedm = xr.open_dataset("/p/projects/megarun/ice_data/Greenland/GRL-8KM/GRL-8KM_TOPO-M17.nc")
vel2 = xr.open_dataset("/p/projects/megarun/ice_data/Greenland/GRL-8KM/GRL-8KM_VEL-J18.nc")
vel=xr.open_dataset("../../FesmData/Joughin2018/output/vel_8km_interp_linear.nc")
rg = xr.open_dataset("/p/projects/megarun/ice_data/Greenland/GRL-8KM/GRL-8KM_REGIONS.nc")
bedm = bedm.assign_coords(xc=sim0.xc, yc=sim0.yc)
rg = rg.assign_coords(xc=sim0.xc, yc=sim0.yc)
vel2 = vel.assign_coords(xc=sim0.xc, yc=sim0.yc)
schu = xr.open_dataset("../../FesmData/Schumacher2018_GIA_GrIS/data/schumacher2018_GR.nc")
# Past data
gowan = xr.open_dataset("../../FesmData/Gowan2023_GAPSLIP_GrIS/output/rsl_dataset_reduced.nc")
leger = xr.open_dataset("../../FesmData/Leger2024_PaleoGris/output/paleogris_8km.nc")
leger["xc"] = sim0.xc # They differ in 0.00006km 
leger["yc"] = sim0.yc
lgm=xr.open_dataset("/p/projects/megarun/luciagu/data/leger2024/lgm.nc")
lgm_leg = lgm.sel(
    xc=slice(-720.00006, 960.00006),
    yc=slice(-3450.0002, -570.0))

lgm_leg = lgm_leg.interp(xc=sim0.xc, yc=sim0.yc, method="linear")

# Ensemble
path_ensemble="../../ensemble_reduced/*"
file_name = "yelmo2D_reduced.nc" 
sim_paths = sorted(glob.glob(path_ensemble))



In [3]:
# Present-day topographic misfit: Reduced chi-squared and MSE for the ice cover 
# ((h_sim - h_obs)/err_obs)².sum()/N 
chired_ens_H = []
chired_ens_vel = []
mse_ens_ice_cover = []
chired_ens_z = []
valid_sim_indices = []   

H_obs, H_obs_err = bedm.H_ice.values, bedm.z_bed_err.values
z_obs, z_obs_err = bedm.z_bed.values, bedm.z_bed_err.values    
mask_bedm = xr.where((bedm.H_ice > 0)&(rg.mask==1.3), 1, 0)
v_obs, v_obs_err = vel.uxy_srf, vel.uxy_err
v_obs_err = v_obs_err.fillna(0)

for i, sim_path in enumerate(sim_paths):
    file_path = os.path.join(sim_path, file_name)
    n_sim = int(os.path.basename(sim_path))
    try:
        yelmo = xr.open_dataset(file_path)
    except FileNotFoundError:
        print(f"[ERROR] No se encontró el archivo: {file_path}. Se omite esta simulación.")
        continue
    
    sim=yelmo.sel(time=0)
    
    # Misfit pd1 H_ice
    model = sim.H_ice.values
    chi_s_red_H = np.mean(((model - H_obs) / np.maximum(H_obs_err, 10))**2)
    chired_ens_H.append(chi_s_red_H)
    
    # Misfit pd3 z_bed
    model = sim.z_bed.values
    chi_s_red_z = np.mean(((model - z_obs) / np.maximum(z_obs_err, 10))**2)
    chired_ens_z.append(chi_s_red_z)
    
    # Misfit pd2 ice cover MSE
    mask_yelmo = xr.where((sim.H_ice > 0)&(rg.mask==1.3), 1, 0)
    diff = (mask_yelmo.values - mask_bedm.values)**2
    mse_area = diff.mean()
    mse_ens_ice_cover.append(mse_area)
    
    # Misfit pd4 uxy_s
    model = sim.uxy_s
    v_obs_err_i = v_obs_err
    v_obs_i = v_obs
    diff1 = ((model - v_obs_i) / np.maximum(v_obs_err_i, 1))**2
    diff = xr.where(diff1.isnull(), 0, diff1)
    chi_s_red_vel = np.mean(diff.values)
    chired_ens_vel.append(chi_s_red_vel)
    
    
    valid_sim_indices.append(n_sim)   

M_pd1 = np.stack(chired_ens_H, axis=0)
M_pd2 = np.stack(mse_ens_ice_cover, axis=0)
M_pd3 = np.stack(chired_ens_z, axis=0)
M_pd4 = np.stack(chired_ens_vel, axis=0)
valid_sim_indices = np.stack(valid_sim_indices, axis=0)


In [5]:
ds_pd = xr.Dataset(
    data_vars={
        "M_pd1": (("sim"), M_pd1),
        "M_pd2": (("sim"), M_pd2),
        "M_pd3": (("sim"), M_pd3),
        "M_pd4": (("sim"), M_pd4),
        },
    coords={
        "sim": np.array(valid_sim_indices)})
misfits_pd = ds_pd.sortby("sim")
misfits_pd.to_netcdf("./scores/misfits_pd.nc")

In [4]:
# Past scores - LGM mask
def sc_lgm(sim0):    
    sc = []
    for t in range(-19000, -15900, 500):
        sim=sim0.sel(time=t)
        mask_gl=xr.where((sim.mask_bed==4)&(rg.mask==1.3),1,0)
        mask_gl_in = xr.where((sim.mask_bed==4)&(lgm_leg.mask==3)&(rg.mask==1.3),1,0)
        mask_gl_out = mask_gl - mask_gl_in
        sc1= mask_gl_out.sum().values/mask_gl.sum().values
        sc.append(sc1)    
    sc=np.array(sc)
    return sc.mean()

sc_lgm_ens = []
valid_sim_indices = []     

for path in sim_paths:
    full_path = f"{path}/{file_name}"
    n_sim = int(os.path.basename(path))
    try:
        with xr.open_dataset(full_path) as sim:
            sc = sc_lgm(sim)
            sc_lgm_ens.append(sc)
            valid_sim_indices.append(n_sim)   
    except FileNotFoundError:
        print(f"Archivo no encontrado en: {full_path}")

sc_lgm_ens = np.array(sc_lgm_ens)
valid_sim_indices = np.array(valid_sim_indices)

ds_lgm = xr.Dataset(
    data_vars={
        "M_pt3": (("sim"), sc_lgm_ens)},
    coords={
        "sim": np.array(valid_sim_indices)})
misfits_lgm = ds_lgm.sortby("sim")
misfits_lgm.to_netcdf("./scores/misfits_pt3_lgm.nc")

In [ ]:
# Past scores - surface elevation change
def score_elev_chi(yelmo,ds,icec):
    ens_elev= yelmo.sel(ice_core=icec).z_srf
    obs_elev= ds.sel(ice_core=icec).z_srf
    sigma=ds.sel(ice_core=icec).error
    chi2 = (ens_elev - obs_elev)**2 / sigma**2
    chi2_red = chi2.mean(dim="time")
    return chi2_red.values

M_pt1g=score_elev_chi(yelev,vint,"grip")
M_pt1n=score_elev_chi(yelev,vint,"ngrip")
M_pt1c=score_elev_chi(yelev,vint,"camp_century")
M_pt1d=score_elev_chi(yelev,vint,"dye3")

ds_pd = xr.Dataset(
    data_vars={
        "M_pt1g": (("sim"), np.array(M_pt1g)),
        "M_pt1n": (("sim"), np.array(M_pt1n)),
        "M_pt1c": (("sim"), np.array(M_pt1c)),
        "M_pt1d": (("sim"), np.array(M_pt1d))},
    coords={
        "sim": np.array(yelev.sim.values)})
misfits_pt1 = ds_pd.sortby("sim")
misfits_pt1.to_netcdf("./scores/misfits_pt1.nc")

In [ ]:
# Past scores - isochrones

chi2_ens = []
chi2_ens3 = []
mse_ens = []
valid_sim_indices = []     
sigma=xr.where(leger.err==0, np.nan, leger.err*1e-3)

sigma2 = xr.where((leger.err == 0)|(leger.err == np.nan), 500, leger.err) * 1e-3
sigma2=sigma2.where(leger.age*1e-3<14)

for path in sim_paths:
    full_path = f"{path}/{file_name}"
    n_sim = int(os.path.basename(path))
    sim = xr.open_dataset(full_path)
    diff = leger.age*1e-3 - sim.isochrone.where(leger.age*1e-3<14)
    chi2 = (diff**2) / (sigma2**2)
    mse = (diff**2)

    chi2_values = np.array(chi2).flatten()
    chi2_clean = chi2_values[~np.isnan(chi2_values)]
    chi_mean = np.mean(chi2_clean)

    mse_ens.append(mse.mean())
    chi2_ens.append(chi_mean)
    valid_sim_indices.append(n_sim)   

    
chi2_ens = np.array(chi2_ens)
mse_ens = np.array(mse_ens)
valid_sim_indices = np.array(valid_sim_indices)
ds_isoc= xr.Dataset(
    data_vars={
        "M_pt2_chi2": (("sim"), chi2_ens),
        "M_pt2_mse": (("sim"), mse_ens)},
    coords={
        "sim": np.array(valid_sim_indices)})
misfits_isoc = ds_isoc.sortby("sim")
misfits_isoc.to_netcdf("./scores/misfits_pt2_isoc.nc")


In [14]:
# Combine all the misfits in one dataset

misfit_lgm = xr.open_dataset("./scores/misfits_pt3_lgm.nc")
misfit_pd = xr.open_dataset("./scores/misfits_pd.nc")
misfit_pt1 = xr.open_dataset("./scores/misfits_pt1.nc")
misfit_pt2 = xr.open_dataset("./scores/misfits_pt2_isoc.nc")

M = xr.merge([misfit_pd, misfit_pt1, misfit_pt2, misfit_lgm])
M.to_netcdf("./scores/misfits.nc")
